# Plantar Thermogram Classification: train on Google Colab

Run each cell from top to bottom (Shift + Enter).

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

In [ ]:
# 1. Check the GPU is on
!nvidia-smi -L

In [ ]:
# 2. Get the code and install requirements
!git clone -q https://github.com/AwaisrazaML/plantar-thermogram-classification.git
%cd plantar-thermogram-classification
!pip install -q -r requirements.txt

## 3. Add the dataset

Download the **Plantar Thermogram Database for the Study of Diabetic Foot Complications** from IEEE DataPort as a zip file. Run the next cell and choose the zip when asked.

Each class must end up as a folder inside `data/` (for example `data/Control Group` and `data/DM Group`). Patient sub-folders inside each class are fine: they are used to keep each patient in one split.

In [ ]:
from google.colab import files
import zipfile, pathlib

uploaded = files.upload()
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('data_raw')

# show the top-level structure so you can point DATA_DIR at the folder holding the class folders
for p in sorted(pathlib.Path('data_raw').glob('*/*'))[:20]:
    print(p)

In [ ]:
# 4. Set this to the folder that directly contains the class folders
DATA_DIR = 'data_raw'  # change if the class folders are one level deeper, e.g. 'data_raw/ThermoDataBase'

import pathlib
print([p.name for p in pathlib.Path(DATA_DIR).iterdir() if p.is_dir()])

In [ ]:
# 5. Train (about 5 to 15 minutes on a T4 GPU)
!python src/train.py --data-dir "$DATA_DIR" --model resnet18 --epochs 30

In [ ]:
# 6. Make the results table and charts
!python src/plot_results.py --output-dir outputs

from IPython.display import Image, display
display(Image('outputs/confusion_matrix.png'))
display(Image('outputs/training_curves.png'))

## 7. Download the results

Send the downloaded `results.zip` to Claude to update the README and your CV.

In [ ]:
!cd outputs && zip -q ../results.zip metrics.json history.json confusion_matrix.png training_curves.png
files.download('results.zip')